In [ ]:
#分析训练集数据得到齐次需求下的相关数，用于Homogeneous RL train.ipynb
import pandas as pd
import numpy as np

# =====================================
# 基本配置
# =====================================
DATA_PATH = r"yellow_tripdata_2025-10.parquet"

ZONES = [237, 161, 236, 186, 162, 230, 142, 239, 163]
zone_to_idx = {z: i for i, z in enumerate(ZONES)}
num_zones = len(ZONES)

START_HOUR = 17
END_HOUR   = 19

SLOT_LEN_MIN = 10                # 10 分钟一个 time slot
SLOTS_PER_HOUR = 60 // SLOT_LEN_MIN
TOTAL_SLOTS = (END_HOUR - START_HOUR) * SLOTS_PER_HOUR  # 12


# Braverman 里选的车数量
N_CARS = 2000

# =====================================
# 读数据
# =====================================
df = pd.read_parquet(DATA_PATH)

df["pickup_dt"]  = pd.to_datetime(df["tpep_pickup_datetime"])
df["dropoff_dt"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# 出行时间（分钟）
df["trip_time_min"] = (
    df["dropoff_dt"] - df["pickup_dt"]
).dt.total_seconds() / 60.0

# 去异常
df = df[(df["trip_time_min"] > 1) & (df["trip_time_min"] < 120)]

# =====================================
# 时间筛选：工作日
# =====================================
df["date"] = df["pickup_dt"].dt.date
df["weekday"] = df["pickup_dt"].dt.weekday

df = df[df["weekday"] < 5]  # 周一到周五


# =====================================
# 17:00–19:00
# =====================================
df["hour"] = df["pickup_dt"].dt.hour
df = df[(df["hour"] >= START_HOUR) & (df["hour"] < END_HOUR)]

# =====================================
# OD 只保留 9 个 zone
# =====================================
df = df[
    df["PULocationID"].isin(ZONES) &
    df["DOLocationID"].isin(ZONES)
]

# =====================================
# 使用的工作日数量
# =====================================
num_days = df["date"].nunique()
print(f"Used weekdays: {num_days}")

# =====================================
# 一、计算 P 矩阵（Braverman 原文方法）
# P_ij = (# orders i→j) / (# orders originating at i)
# =====================================
od_counts = (
    df
    .groupby(["PULocationID", "DOLocationID"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=ZONES, columns=ZONES, fill_value=0)
)

origin_counts = od_counts.sum(axis=1)

P_matrix = od_counts.div(origin_counts, axis=0).fillna(0)

# =====================================
# 二、计算 λ_i（Braverman 方法）
# =====================================
# 每个 time slot（10 分钟）到达 region i 的平均订单数
# 然后除以 N
# =====================================

# 到达 region i 的订单
arrival_counts = (
    df
    .groupby("DOLocationID")
    .size()
    .reindex(ZONES, fill_value=0)
)

# 每天 17–19 点一共 TOTAL_SLOTS 个 time slot
avg_arrivals_per_slot = arrival_counts / (num_days * TOTAL_SLOTS)

lambda_i = avg_arrivals_per_slot / N_CARS

# 转成向量
lambda_vec = lambda_i.values
lambda_total = lambda_vec.sum()

# =====================================
# 三、时间矩阵 T_ij（单位：分钟）
# =====================================
T_matrix = (
    df
    .groupby(["PULocationID", "DOLocationID"])["trip_time_min"]
    .mean()
    .unstack(fill_value=0)
    .reindex(index=ZONES, columns=ZONES, fill_value=0)
)

# =====================================
# 输出结果
# =====================================
print("\nλ_i (Braverman definition):")
for z, val in zip(ZONES, lambda_vec):
    print(f"Region {z}: λ_i = {val:.6f}")

print(f"\nλ_total = {lambda_total:.6f}")

print("\nP matrix:")
print(P_matrix)

print("\nTime matrix T_ij (minutes):")
print(T_matrix)


Used weekdays: 23

λ_i (Braverman definition):
Region 237: λ_i = 0.024857
Region 161: λ_i = 0.011875
Region 236: λ_i = 0.023431
Region 186: λ_i = 0.005790
Region 162: λ_i = 0.008067
Region 230: λ_i = 0.010641
Region 142: λ_i = 0.015862
Region 239: λ_i = 0.012096
Region 163: λ_i = 0.008873

λ_total = 0.121493

P matrix:
DOLocationID       237       161       236       186       162       230  \
PULocationID                                                               
237           0.208592  0.108899  0.298686  0.019514  0.083720  0.049807   
161           0.227981  0.096316  0.187767  0.081521  0.052825  0.084004   
236           0.325079  0.074293  0.210677  0.015343  0.058412  0.033019   
186           0.082594  0.150199  0.070052  0.042521  0.109820  0.255430   
162           0.218182  0.093013  0.194290  0.088956  0.071675  0.101728   
230           0.110667  0.109905  0.103810  0.119810  0.066095  0.112381   
142           0.160912  0.085743  0.135635  0.023955  0.058318  0.09929